(mmm_multi_arm_geolift_calibration)=
# MMM Calibration with Multi-Arm Geo Lift Tests

A retailer wants to learn how search advertising affects revenue without increasing its total test budget. It increases search spend in 16 markets, decreases it in 16, and leaves 16 unchanged. The unchanged markets help estimate what would have happened without the intervention. The up and down arms then constrain **different parts of the search response curve**, even when their budget changes cancel.

We use [CausalPy](https://causalpy.readthedocs.io/en/latest/) to estimate the signed effects and a hierarchical geo MMM to compare what we learn with and without those effects. Both MMMs already adjust for the common weekly shock that drives budgets and revenue.

:::{dropdown} Reproduce the notebook environment

Run from a repository checkout and use its Python 3.12 kernel. Install Graphviz so that the `dot` executable is available for the causal diagram.

```bash
uv sync --locked --python 3.12 --group geolift-notebook
uv run --no-sync python -m ipykernel install --prefix .venv --name python3
```
:::

In [ ]:
import logging

import arviz as az
import causalpy as cp
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pymc.dims as pmd
import pytensor.xtensor as ptx
import xarray as xr
from graphviz import Digraph
from IPython.display import HTML, SVG, display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, StrMethodFormatter
from pymc_extras.prior import Prior

from pymc_marketing.mmm import MMM, ControlMuEffect, LogisticSaturation, NoAdstock
from pymc_marketing.mmm.data_conversion import to_mmm_dataset
from pymc_marketing.mmm.scaling import FixedScaling, Scaling
from pymc_marketing.mmm.transformers import logistic_saturation

%matplotlib inline
# Store Matplotlib figures at double pixel density for sharp documentation output.
%config InlineBackend.figure_format = "retina"
logging.getLogger("pymc").setLevel(logging.WARNING)
FIG_WIDTH, FIG_HEIGHT = 10, 3.5
COLOR_UP, COLOR_DOWN, COLOR_CONTROL = "#0072B2", "#D55E00", "#777777"
COLOR_NET = "#7B3294"
COLOR_DONOR = "#87929D"
CHANNEL_COLORS = {"search": "#0072B2", "display": "#009E73"}
ARM_COLORS = {"up": COLOR_UP, "down": COLOR_DOWN, "control": COLOR_CONTROL}
plt.rcParams.update(
    {
        "figure.figsize": (FIG_WIDTH, FIG_HEIGHT),
        "font.size": 12,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.constrained_layout.use": True,
    }
)
USD = StrMethodFormatter("${x:,.0f}")
SEED = sum(map(ord, "Signed multi-arm geo calibration"))
streams = np.random.SeedSequence(SEED).spawn(5)
rng_parameters, rng_spend, rng_history, rng_experiment, rng_assignment = [
    np.random.default_rng(s) for s in streams
]
SCALE = 100_000.0
channels = ["search", "display"]

In [ ]:
# CI uses shorter real fits to check integration.
ci_smoke = False

In [ ]:
dag = Digraph("calibration_dgp")
dag.attr(rankdir="TB", bgcolor="white", nodesep="0.3", ranksep="0.5", pad="0.15")
dag.attr("node", shape="box", style="rounded", fontname="Helvetica", fontsize="14")
dag.attr("edge", color="#444444", fontname="Helvetica", fontsize="11")
dag.node("assignment", "Arm assignment\n(up / down / control)")
dag.node("budget", "Local budget\nfactor", style="rounded,dashed")
dag.node("demand", "Latent demand", style="rounded,dashed")
dag.node("search", "Search spend")
dag.node("display", "Display spend")
dag.node("revenue", "Revenue in geo g")
dag.edge("assignment", "search")
dag.edge("budget", "search")
dag.edge("budget", "display")
dag.edge("demand", "search")
dag.edge("demand", "revenue")
dag.edge("search", "revenue")
dag.edge("display", "revenue")
display(SVG(dag.pipe(format="svg")))

**Caption:** Causal structure for simulated geo-level revenue, conditional on geo-specific baseline budgets, baseline revenue, response parameters, and observed seasonality. Dashed nodes are unobserved demand and local budget drivers. Independent spend and observation noise are omitted.

Demand affects search budgets and revenue. Local budgets drive both channels. These processes continue throughout the example. During the experiment, randomized arm assignment adds a signed increment to ordinary search budgets; it does not alter demand, display budgets, or revenue response parameters. This is an intervention on the spend increment, rather than replacement of the entire search-budget rule. Selected controls receive no increment and supply the donor pool; up/down doses are randomized among the remaining geos.

## Setup and data-generating process

The retailer operates in 48 simulated US markets. We use 136 weeks of history followed by an eight-week search experiment. In both periods, weekly revenue for geo $g$ and week $t$ is

$$
\begin{aligned}
Y_{gt} &= a_g+\sum_c f_{gc}(X_{gtc})+s_t+10{,}000 D_t+\epsilon_{gt},\\
f_{gc}(x) &= 100{,}000\beta_{gc}\tanh\!\left(\frac{\lambda_{gc}x}{2\times100{,}000}\right).
\end{aligned}
$$


The sum runs over search and display. Baseline revenue $a_g$ varies by geo, seasonality $s_t$ is shared, and independent observation noise has SD 1,000 USD/week. Common demand $D_t$ raises both search budgets and non-media revenue. The channels also share local weekly budget shocks, which create correlation even though display budgets are independent of $D_t$.

The ordinary budget rule continues through the experiment. Only the assigned search increment is new. Display and control-market spend follow their ordinary paths.

In [ ]:
geos = (
    "AL AZ AR CA CO CT DE FL GA ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT "
    "NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY"
).split()
geos = sorted(geos)
G = len(geos)
# Select an illustrative donor pool before randomizing the active doses.
donor_geos = geos[::3]
dates = pd.date_range("2022-01-03", periods=144, freq="W-MON")
train_dates = dates[:136]
estimation_dates = dates
experiment_start, experiment_end = dates[136], dates[-1]
is_intervention = dates >= experiment_start
window = dates[is_intervention]
true_lam = rng_parameters.lognormal(np.log([4.0, 3.0]), 0.12, (G, 2))
true_beta = rng_parameters.lognormal(np.log([0.6, 0.5]), 0.12, (G, 2))
true_intercept = rng_parameters.uniform(0.50, 0.70, G)
true_intercept[[geos.index(g) for g in donor_geos]] = np.linspace(
    0.20, 1.00, len(donor_geos)
)
base_spend = rng_spend.uniform(32_000, 48_000, (G, 2))
# Market properties are fixed before treatment assignment. Controls are selected,
# not randomized; only the 32 experimental markets receive randomized up/down doses.
experimental_geos = [g for g in geos if g not in donor_geos]
assigned = rng_assignment.permutation(experimental_geos).tolist()
arm_size = len(geos) // 3
arms = {
    "up": sorted(assigned[:arm_size]),
    "down": sorted(assigned[arm_size:]),
    "control": donor_geos,
}
geo_arm = {g: arm for arm, members in arms.items() for g in members}


# Smooth, shared latent demand drives search budgets and non-media revenue.
# Fitted MMMs estimate a shared baseline without observing this index.
def smooth_process(shape, persistence=0.85):
    values = rng_spend.normal(size=shape)
    for t in range(1, shape[0]):
        values[t] = (
            persistence * values[t - 1] + np.sqrt(1 - persistence**2) * values[t]
        )
    return (values - values.mean(axis=0)) / values.std(axis=0)


latent_demand = smooth_process((144,))
local_budget = rng_spend.normal(size=(len(dates), G, 1))
baseline = base_spend[None, :, :] * (
    1 + latent_demand[:, None, None] * np.array([0.18, 0.0]) + 0.08 * local_budget
) + rng_spend.normal(0, 1_500, (len(dates), G, 2))
historical_spend = baseline[~is_intervention]
DEMAND_EFFECT = 10_000.0  # Direct revenue change (USD/week) per SD of demand.
planned = baseline.copy()
for arm, sign in [("up", 1), ("down", -1)]:
    planned[np.ix_(is_intervention, [geos.index(g) for g in arms[arm]], [0])] += (
        sign * 8_000
    )
realized = baseline + 0.9 * (planned - baseline)
season_controls = np.column_stack(
    [np.sin(2 * np.pi * np.arange(144) / 52), np.cos(2 * np.pi * np.arange(144) / 52)]
)
true_gamma = np.array([0.06, 0.03])


def media_response(spend):
    x = ptx.as_xtensor(spend / SCALE, dims=("time", "geo", "channel"))
    lam = ptx.as_xtensor(true_lam, dims=("geo", "channel"))
    saturated = (
        logistic_saturation(x, lam=lam).transpose("time", "geo", "channel").eval()
    )
    return (true_beta * saturated).sum(axis=-1) * SCALE


non_media = (
    SCALE * (true_intercept + season_controls @ true_gamma[:, None])
    + DEMAND_EFFECT * latent_demand[:, None]
)
mu_baseline = non_media + media_response(baseline)
mu_experiment = non_media + media_response(realized)
mu_history = mu_baseline[~is_intervention]
y_history = mu_history + rng_history.normal(0, 1_000, mu_history.shape)
y_experiment = mu_experiment[is_intervention] + rng_experiment.normal(
    0, 1_000, (is_intervention.sum(), G)
)
true_lift = pd.Series(
    (mu_experiment - mu_baseline)[is_intervention].mean(axis=0), index=geos
)

In [ ]:
def long_features(spend, selected_dates, controls):
    frame = pd.DataFrame(spend.reshape(-1, 2), columns=channels)
    frame["baseline_input"] = 1.0
    frame["date"] = np.repeat(selected_dates, G)
    frame["geo"] = np.tile(geos, len(selected_dates))
    frame[["annual_sin", "annual_cos"]] = np.repeat(controls, G, axis=0)
    return frame


# Both stages use the same 136 pre-intervention weeks.
training_spend = realized[~is_intervention]
X_train = long_features(training_spend, train_dates, season_controls[:136])
y_train = pd.Series(y_history.ravel(), name="revenue")
experiment_revenue = pd.DataFrame(
    np.vstack([y_history, y_experiment]), index=dates, columns=geos
)
spend_columns = pd.MultiIndex.from_product([geos, channels], names=["geo", "channel"])
spend_baseline = pd.DataFrame(
    baseline.reshape(len(dates), -1), index=dates, columns=spend_columns
)
spend_realized = pd.DataFrame(
    realized.reshape(len(dates), -1), index=dates, columns=spend_columns
)
for frame in (experiment_revenue, spend_baseline, spend_realized):
    frame.attrs["unit"] = "USD/week"
np.testing.assert_equal(len(set(geos)), 48)
np.testing.assert_equal([len(arms[a]) for a in arms], [16, 16, 16])
np.testing.assert_equal(len(train_dates.intersection(estimation_dates)), 136)
if train_dates.max() >= experiment_start:
    raise ValueError("MMM target dates must precede the intervention.")
np.testing.assert_array_equal(realized[:, :, 1], baseline[:, :, 1])
np.testing.assert_array_equal(
    realized[:, [geos.index(g) for g in arms["control"]]],
    baseline[:, [geos.index(g) for g in arms["control"]]],
)
for values in (historical_spend, baseline, realized, y_history, y_experiment):
    if not np.isfinite(values).all() or values.min() <= 0:
        raise ValueError("The simulation must have finite positive spend and revenue.")


def as_mmm_data(features, target):
    return to_mmm_dataset(
        features,
        target,
        date_column="date",
        dims=("geo",),
        channel_columns=channels,
        control_columns=["annual_sin", "annual_cos"],
        extra_vars=["baseline_input"],
        target_column="revenue",
    )


mmm_training_data = as_mmm_data(X_train, y_train)

We designate every third market as a donor and assign its baseline revenue from a wider range than in the active markets. This creates a donor pool with a range of baseline revenues before we randomize the remaining markets between up and down doses. We check pre-intervention fit in Step 1.

The figure indexes each geo's revenue to 100 in the last pre-intervention week and averages those indices by arm. Both fits use the original outcomes.

In [ ]:
indexed_revenue = experiment_revenue.div(experiment_revenue.loc[train_dates[-1]]) * 100
np.testing.assert_allclose(indexed_revenue.loc[train_dates[-1]], 100)
fig, (geo_ax, arm_ax) = plt.subplots(
    2, 1, figsize=(FIG_WIDTH, FIG_HEIGHT * 1.75), sharex=True
)
for arm, members in arms.items():
    indexed_arm = indexed_revenue[members]
    geo_ax.plot(dates, indexed_arm, color=ARM_COLORS[arm], alpha=0.24, linewidth=0.8)
    arm_ax.plot(
        dates, indexed_arm.mean(axis=1), color=ARM_COLORS[arm], label=arm.title()
    )
for ax in (geo_ax, arm_ax):
    ax.axhline(100, color=COLOR_CONTROL, linestyle=":", linewidth=1)
    ax.axvline(experiment_start, color="black", linestyle="--", linewidth=1)
    ax.set_ylabel("Revenue index")
geo_ax.set_title("Individual markets")
arm_ax.set_title("Average by arm")
arm_ax.legend(ncol=3)
arm_ax.set_xlabel("Week")
plt.show()

**Caption:** Indexed revenue for individual markets and arm averages. The vertical line marks the first week of assigned search changes.

The variable-budget history contains correlated channels because they share local budget changes. Search also tracks latent demand; display does not. The association is partly noncausal: demand also raises revenue without passing through media. The next section creates spend changes by assignment instead of following demand; unchanged markets retain the common demand signal used to estimate untreated revenue.

### A spend-neutral intervention

Equal planned increases and decreases all receive 90% delivery, so the delivered changes cancel by construction. The table checks the implemented spend paths against that design. Actual delivery can differ by market, so verify neutrality from observed delivery in an applied test.

In [ ]:
accounting = pd.DataFrame(index=["up", "down", "control"])
for label, panel in [("planned", planned), ("delivered", realized)]:
    accounting[label] = [
        (panel - baseline)[is_intervention][:, [geos.index(g) for g in arms[a]], 0]
        .sum(axis=1)
        .mean()
        for a in accounting.index
    ]
accounting.loc["net"] = accounting.sum()
for panel in (planned, realized):
    np.testing.assert_allclose(
        (panel - baseline)[is_intervention, :, 0].sum(axis=1), 0, atol=1e-8
    )
np.testing.assert_equal((baseline[~is_intervention].std(axis=0) > 0).all(), True)
np.testing.assert_equal((baseline[is_intervention].std(axis=0) > 0).all(), True)
np.testing.assert_allclose(
    np.ptp((realized - baseline)[is_intervention], axis=0), 0, atol=1e-8
)
# Choose all example markets from baseline spend, before estimating any effects.
ranked_geos = {
    arm: sorted(members, key=lambda g: base_spend[geos.index(g), 0])
    for arm, members in arms.items()
}
representatives = {arm: members[arm_size // 2] for arm, members in ranked_geos.items()}
example_ranks = np.linspace(0, arm_size - 1, 4, dtype=int).tolist()
curve_examples = [
    (arm, ranked_geos[arm][rank]) for arm in arms for rank in example_ranks
]
return_examples = [
    (arm, ranked_geos[arm][rank]) for rank in example_ranks for arm in ("up", "down")
]
display_example_ranks = np.linspace(0, arm_size - 1, 3, dtype=int).tolist()
display_ranked_geos = {
    arm: sorted(members, key=lambda g: base_spend[geos.index(g), 1])
    for arm, members in arms.items()
}
display_examples = [
    (arm, display_ranked_geos[arm][rank])
    for arm in arms
    for rank in display_example_ranks
]
display(accounting.style.format("${:,.0f}"))

## Step 1: estimate experimental lift with synthetic controls

CausalPy learns donor weights from the 136 pre-intervention weeks. It uses the unchanged markets to predict what revenue would have been without the intervention, then subtracts that counterfactual from observed revenue. One joint fit estimates both active arms against the same donor pool.

Fixed donor weights approximate the shared demand signal. Markets have different nonlinear media responses and their own budget shocks, so a weighted donor mix will still mispredict some treated outcomes.

In [ ]:
sc_fit_kwargs = dict(
    tune=40 if ci_smoke else 2_000,
    draws=40 if ci_smoke else 2_000,
    chains=2,
    cores=2,
    target_accept=0.95,
    nuts_sampler="nutpie",
    random_seed=SEED + 10,
    progressbar=False,
)
mmm_fit_kwargs = dict(
    tune=40 if ci_smoke else 3_000,
    draws=40 if ci_smoke else 3_000,
    chains=2,
    cores=2,
    nuts_sampler="nutpie",
    random_seed=SEED + 20,
    progressbar=False,
    nuts={"target_accept": 0.9, "maxdepth": 4}
    if ci_smoke
    else {
        "target_accept": 0.99,
        "adaptation": "low_rank",
        "mass_matrix_eigval_cutoff": 3,
    },
)
result = cp.UpDownGeoLift(
    experiment_revenue,
    treatment_time=experiment_start,
    arms=arms,
    model=cp.pymc_models.SoftmaxWeightedSumFitter(sample_kwargs=sc_fit_kwargs),
).fit()

# Use predictive outcomes, not only uncertainty in the fitted conditional mean.
pre_counterfactual = result.model.predict(
    result.pre_design["control"]
).posterior_predictive["y_hat"]
post_counterfactual = result.model.predict(
    result.post_design["control"]
).posterior_predictive["y_hat"]
pre_impact = result.pre_design["treated"] - pre_counterfactual
post_impact = result.post_design["treated"] - post_counterfactual

### Carry residual uncertainty into the MMM

An eight-week mean of CausalPy's conditional-mean impacts reflects donor-weight uncertainty. It leaves out the treated market's own residual variation. With independent weekly residuals under the fitted counterfactual model, add the posterior mean squared residual scale divided by eight to the variance of the conditional effect draws. The square root is the SD of the mean weekly effect. Weekly figures and arm totals use posterior-predictive draws, which already include the residual term.

The table reports spend and lift in USD/week. `x` and `delta_x` are window means for orientation. Step 2 uses the full weekly spend paths to evaluate the nonlinear response. Here the simulator supplies business-as-usual spend. In a real experiment, record the planned no-test budget before assignment or estimate it credibly. This likelihood treats that path as fixed. `delta_y` and `sigma` summarize the same eight-week mean impact.

CausalPy's scalar `to_mmm_lift` exporter requires stable spend and rejects lift estimates whose signs disagree with delivered spend. To retain a noisy zero or opposite-sign estimate, use `aggregate_draws` and build the rows as shown below. Inspect sign disagreements for design or counterfactual problems. For a credible estimate, pass the observed sign and uncertainty to the signed Normal likelihood. Filtering on the estimated sign selects errors in the expected direction. Persistent or precise opposite-sign effects can also challenge the MMM's assumption of a monotone response.

In [ ]:
# The fitted conditional mean leaves out the treated geo's residual noise.
conditional_effect = result.aggregate_draws(
    start=experiment_start, end=experiment_end, aggregate="mean"
)["geo"]
sample_dims = ["chain", "draw"]
residual_sd = result.model.idata.posterior["y_hat_sigma"]
mean_residual_variance = (residual_sd**2).mean(sample_dims) / len(window)
lift_sd = np.sqrt(conditional_effect.var(sample_dims) + mean_residual_variance)

# Draws for intervals and arm totals come from the same predictive model.
effect_draws = post_impact.sel(obs_ind=window).mean("obs_ind")
calibration_geos = conditional_effect.treated_units.to_numpy()
effect_draws = effect_draws.sel(treated_units=calibration_geos)
search_baseline = spend_baseline.loc[window, [(g, "search") for g in calibration_geos]]
search_delivered = spend_realized.loc[window, [(g, "search") for g in calibration_geos]]
lift_rows = pd.DataFrame(
    {
        "geo": calibration_geos,
        "arm": [geo_arm[g] for g in calibration_geos],
        "channel": "search",
        "x": search_baseline.mean().to_numpy(),
        "delta_x": (search_delivered - search_baseline).mean().to_numpy(),
        "delta_y": conditional_effect.mean(sample_dims).to_numpy(),
        "sigma": lift_sd.sel(treated_units=calibration_geos).to_numpy(),
        "window_start": experiment_start,
        "window_end": experiment_end,
        "n_periods": len(window),
        "spend_unit": "USD/week",
        "outcome_unit": "USD/week",
    }
)
np.testing.assert_equal(len(lift_rows), 32)
np.testing.assert_equal(set(lift_rows.geo), set(arms["up"] + arms["down"]))
if (
    not np.isfinite(lift_rows[["delta_y", "sigma"]]).all().all()
    or (lift_rows.sigma <= 0).any()
):
    raise ValueError("Calibration effects and SDs must be finite, with positive SDs.")
display(
    lift_rows[["geo", "arm", "channel", "x", "delta_x", "delta_y", "sigma"]]
    .groupby("arm", sort=False)
    .head(2)
    .style.format({c: "{:,.1f}" for c in ["x", "delta_x", "delta_y", "sigma"]})
)

In [ ]:
pre_residual = result.result.impact_pre
pre_rmse = np.sqrt((pre_residual.mean(["chain", "draw"]) ** 2).mean("obs_ind"))
pre_table = pre_rmse.to_series().rename("Pre-fit RMSE (USD/week)").to_frame()
pre_table["arm"] = [geo_arm[g] for g in pre_table.index]
pre_table = pre_table.sort_values(["arm", "Pre-fit RMSE (USD/week)"])
display(
    HTML(
        '<details class="admonition note">'
        "<summary><strong>Optional diagnostic: pre-intervention donor fit</strong></summary>"
        "<p>RMSE summarizes the pre-intervention residuals for every treated market, in USD/week. "
        "A poor fit is a reason to investigate the donor model. A good in-sample fit does not "
        "establish that post-intervention counterfactual assumptions hold.</p>"
        + pre_table.style.format({"Pre-fit RMSE (USD/week)": "${:,.0f}"}).to_html()
        + "</details>"
    )
)

**Pre-period diagnostic.** Inspect residual RMSE before using a donor fit. The time series illustrate two markets selected by median baseline search spend.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(FIG_WIDTH, FIG_HEIGHT * 2), sharex=True)
for col, arm in enumerate(["up", "down"]):
    g = representatives[arm]
    color = ARM_COLORS[arm]
    pre = pre_impact.sel(treated_units=g)
    post = post_impact.sel(treated_units=g)
    impacts = xr.concat([pre, post], dim="obs_ind").transpose(
        "chain", "draw", "obs_ind"
    )
    observed = experiment_revenue[g]
    cf = (
        xr.DataArray(
            observed.to_numpy(), dims="obs_ind", coords={"obs_ind": observed.index}
        )
        - impacts
    )
    for row, draws, label in [
        (0, cf, "Counterfactual mean"),
        (1, impacts, "Signed impact"),
    ]:
        ax = axes[row, col]
        bounds = draws.quantile([0.03, 0.97], dim=["chain", "draw"])
        ax.fill_between(
            estimation_dates,
            bounds.isel(quantile=0),
            bounds.isel(quantile=1),
            color=color,
            alpha=0.2,
        )
        ax.plot(
            estimation_dates,
            draws.mean(["chain", "draw"]),
            color=color,
            linestyle="--" if row == 0 else "-",
            label=label,
        )
        ax.axvline(experiment_start, color="black", linestyle="--")
        ax.yaxis.set_major_formatter(USD)
        ax.set_ylabel("Revenue (USD/week)" if row == 0 else "Impact (USD/week)")
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=4))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
        ax.legend(fontsize=11)
    axes[0, col].plot(
        estimation_dates, observed, color="black", linewidth=1, label="Observed"
    )
    axes[0, col].set_title(f"{g} · {arm}")
    axes[0, col].legend(fontsize=11)
    axes[1, col].axhline(0, color="black", linestyle="--")
    axes[1, col].set_xlabel("Week")
plt.show()

**Caption:** Observed revenue, synthetic counterfactual, and signed weekly impact for median-baseline-spend up/down geos. Bands are 94% equal-tailed posterior-predictive intervals. The vertical dashed line marks intervention. These impact intervals condition on observed donor outcomes.

**Stakeholder question: How much weekly revenue did the search intervention add or remove in each treated geo?**

The next plot compares CausalPy's posterior estimates of each geo's eight-week mean effect with the known true effects, so we can inspect signs, magnitudes, and uncertainty.

In [ ]:
geo_effects = lift_rows.set_index("geo")[["arm"]].copy()
geo_effects["mean"] = lift_rows.set_index("geo").delta_y
geo_effects["lower_94"] = effect_draws.quantile(0.03, dim=["chain", "draw"]).to_numpy()
geo_effects["upper_94"] = effect_draws.quantile(0.97, dim=["chain", "draw"]).to_numpy()
geo_effects["truth"] = true_lift.reindex(geo_effects.index)
fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT * 2.5))
for i, (_g, row) in enumerate(geo_effects.iterrows()):
    color = ARM_COLORS[row.arm]
    ax.plot([row.lower_94, row.upper_94], [i, i], color=color)
    ax.scatter(row["mean"], i, color=color, s=20)
    ax.scatter(row.truth, i, color=color, marker="x", s=30)
ax.legend(
    handles=[
        Line2D(
            [], [], marker="o", linestyle="none", color="black", label="Posterior mean"
        ),
        Line2D([], [], color="black", label="94% equal-tailed interval"),
        Line2D(
            [],
            [],
            marker="x",
            linestyle="none",
            color="black",
            label="True mean effect",
        ),
        Patch(facecolor=COLOR_UP, label="Up arm"),
        Patch(facecolor=COLOR_DOWN, label="Down arm"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 1.01),
    ncol=3,
    fontsize=10,
)
ax.set_yticks(range(len(geo_effects)), geo_effects.index)
ax.invert_yaxis()
ax.axvline(0, color="black", linestyle="--")
ax.xaxis.set_major_formatter(USD)
ax.set_xlabel("Mean weekly search lift (USD/week)")
ax.set_ylabel("Geo")
plt.show()

**Caption:** Estimated mean weekly lift (dots), 94% predictive intervals (segments), and simulated truth (crosses).

In [ ]:
truth_by_geo = xr.DataArray(
    true_lift.loc[calibration_geos].to_numpy(),
    dims="treated_units",
    coords={"treated_units": calibration_geos},
)
lift_diagnostics = []
for label, draws, sd in [
    ("Conditional mean", conditional_effect, conditional_effect.std(sample_dims)),
    ("Including residuals", effect_draws, lift_sd),
]:
    lower, upper = draws.quantile([0.03, 0.97], dim=sample_dims)
    lift_diagnostics.append(
        {
            "Effect draws": label,
            "Median SD (USD/week)": float(sd.median()),
            "Truths in 94% interval (of 32)": int(
                ((lower <= truth_by_geo) & (truth_by_geo <= upper)).sum()
            ),
            "SD of standardized errors": float(
                ((conditional_effect.mean(sample_dims) - truth_by_geo) / sd).std()
            ),
        }
    )
display(pd.DataFrame(lift_diagnostics).set_index("Effect draws").round(2))

In this simulation the 94% intervals contain 15 of the 32 true effects using conditional-mean draws and 27 after adding residual variation. The wider intervals better reflect the observed errors.

### Does spend neutrality imply revenue neutrality?

Sum the aligned weekly effect draws within each arm and across both active arms to see the net revenue effect.

In [ ]:
# Sum corresponding joint draws, retaining the dependence represented by the fit.
arm_totals = [
    effect_draws.sel(treated_units=arms[arm]).sum("treated_units")
    for arm in ["up", "down"]
]
revenue_totals = xr.concat(
    [*arm_totals, effect_draws.sum("treated_units")],
    dim=pd.Index(["up", "down", "net"], name="summary"),
)
true_totals = [true_lift.loc[arms[arm]].sum() for arm in ["up", "down"]]
true_totals.append(true_lift.loc[arms["up"] + arms["down"]].sum())
means = revenue_totals.mean(["chain", "draw"])
intervals = revenue_totals.quantile([0.03, 0.97], dim=["chain", "draw"])
fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT * 1.3))
for i, (label, color, expected_total) in enumerate(
    zip(
        ["up", "down", "net"],
        [COLOR_UP, COLOR_DOWN, COLOR_NET],
        true_totals,
        strict=True,
    )
):
    mean = float(means.sel(summary=label))
    lower, upper = intervals.sel(summary=label).values
    ax.bar(i, mean, color=color, alpha=0.3, width=0.55)
    ax.errorbar(
        i, mean, yerr=[[mean - lower], [upper - mean]], fmt="o", color=color, capsize=6
    )
    ax.scatter(i, expected_total, marker="x", color=color, s=70, zorder=4)
ax.legend(
    handles=[
        Patch(facecolor="gray", alpha=0.3, label="Estimated total (bar)"),
        Line2D([], [], marker="o", color="gray", label="Posterior mean (dot)"),
        ax.errorbar(
            [],
            [],
            yerr=[],
            fmt="none",
            color="gray",
            capsize=6,
            label="94% equal-tailed interval",
        ),
        Line2D([], [], marker="x", linestyle="none", color="gray", label="True total"),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, 1.24),
    ncol=2,
    fontsize=10,
)
ax.set_xticks(range(3), ["Up · 16 markets", "Down · 16 markets", "Net · 32 markets"])
ax.axhline(0, color="black", linestyle="--")
ax.yaxis.set_major_formatter(USD)
ax.set_ylabel("Total revenue effect (USD/week)")
plt.show()

**Caption:** Mean total weekly revenue effects (bars/dots), 94% equal-tailed intervals (whiskers), and true totals (crosses), for each active arm and their net effect. Totals sum aligned geo-level predictive effect draws before computing intervals, retaining the dependence available in those draws. Counterfactual model limitations still apply.

The known response curves determine whether up-market gains offset down-market losses. Equal spend changes do not force them to match. More deliberate market assignment or dose choices could target revenue neutrality, but would require pre-experiment response information, credible counterfactuals, and statistical power. We do not optimize that allocation here.

## Step 2: calibrate a hierarchical geo MMM

Week effects can absorb the common additive demand shock in this simulation. We fit a regularized version: one zero-sum weekly revenue effect shared across all markets, with a prior scale of USD 15,000/week. It is estimated from the historical panel without the simulated demand index. Comparing **two models with the same weekly adjustment** isolates what calibration adds to the time-adjusted baseline. A shared week effect would be insufficient for an unobserved demand shock with geo-specific effects.

Both fits use the same 136 historical outcomes, and both partially pool log-curvature (`lam`) and log-amplitude (`beta`) across geos within each channel. The log-population mean priors are centered at the generating channel medians (curvature 4 and 3, amplitude 0.6 and 0.5). The intercept prior centers on USD 60,000, the midpoint of the simulated active-market baselines. These are advantages of knowing the simulator. On real data, justify and check those centers separately. Population spreads are estimated.

Only the second fit receives the experimental lift summaries. It has one constraint per treated geo on a mean difference, so amplitude and curvature can still trade off. Both fits use Nutpie low-rank adaptation with 3,000 tuning and 3,000 retained draws per chain.

In [ ]:
def make_mmm():
    saturation_priors = {
        "lam": Prior(
            "Normal",
            transform="exp",
            mu=Prior(
                "Normal",
                mu=xr.DataArray(
                    np.log([4.0, 3.0]), dims="channel", coords={"channel": channels}
                ),
                sigma=0.5,
                dims="channel",
            ),
            sigma=Prior("HalfNormal", sigma=0.3, dims="channel"),
            dims=("geo", "channel"),
            centered=False,
        ),
        "beta": Prior(
            "Normal",
            transform="exp",
            mu=Prior(
                "Normal",
                mu=xr.DataArray(
                    np.log([0.6, 0.5]), dims="channel", coords={"channel": channels}
                ),
                sigma=0.5,
                dims="channel",
            ),
            sigma=Prior("HalfNormal", sigma=0.3, dims="channel"),
            dims=("geo", "channel"),
            centered=False,
        ),
    }
    mmm = MMM(
        date_column="date",
        target_column="revenue",
        channel_columns=channels,
        dims=("geo",),
        control_columns=["annual_sin", "annual_cos"],
        adstock=NoAdstock(l_max=1),
        saturation=LogisticSaturation(priors=saturation_priors),
        scaling=Scaling(
            channel=FixedScaling(dims=(), value=SCALE),
            target=FixedScaling(dims=(), value=SCALE),
        ),
        model_config={
            "intercept": Prior("Normal", mu=0.6, sigma=0.4, dims="geo"),
            "gamma_control": Prior("Normal", mu=0, sigma=0.1, dims="control"),
            "likelihood": Prior(
                "Normal", sigma=Prior("HalfNormal", sigma=0.15, dims="geo")
            ),
        },
    )
    # One latent weekly effect is shared by all geos (0.15 scaled = USD 15,000).
    # The input is constant; no simulated demand values enter the fitted model.
    mmm.add_mu_effect(
        ControlMuEffect(
            data_vars=["baseline_input"],
            prefix="weekly_baseline",
            prior=Prior("ZeroSumNormal", sigma=0.15, dims="date", core_dims="date"),
        )
    )
    return mmm

In [ ]:
# Verify the media response and outcome scales independently of omitted demand.
truth_mmm = make_mmm()
X_baseline = long_features(baseline, estimation_dates, season_controls)
X_realized = long_features(realized, estimation_dates, season_controls)
truth_predictions = []
for features in (X_baseline, X_realized):
    truth_mmm.build_model(as_mmm_data(features, pd.Series(np.ones(len(features)))))
    fixed = pm.do(
        truth_mmm.model,
        {
            "saturation_lam": true_lam,
            "saturation_beta": true_beta,
            "intercept_contribution": true_intercept,
            "gamma_control": true_gamma,
            "weekly_baseline_baseline_input_coef": (
                DEMAND_EFFECT * latent_demand / SCALE
            ),
        },
    )
    truth_predictions.append(
        pm.draw(fixed["mu"].transpose("date", "geo"), random_seed=SEED) * SCALE
    )
np.testing.assert_allclose(truth_predictions[0], mu_baseline, rtol=1e-10)
np.testing.assert_allclose(truth_predictions[1], mu_experiment, rtol=1e-10)
np.testing.assert_allclose(
    (truth_predictions[1] - truth_predictions[0])[is_intervention].mean(axis=0),
    true_lift,
    atol=1e-8,
)
del truth_mmm
print("Fixed-truth media response and shared baseline reproduce the simulator.")

The fixed-truth check verifies the model's scales and temporal contrast. The fitted graphs receive a constant for the shared week effect and learn its coefficients from history.

### Attach the signed temporal contrast

The likelihood compares each estimated lift with the mean of the eight weekly response differences, giving noisier estimates less weight. Attach it after building the MMM so that fitting uses the augmented graph.

In [ ]:
mmm_uncalibrated = make_mmm()
mmm_uncalibrated.build_model(mmm_training_data)
idata_uncalibrated = mmm_uncalibrated.fit(mmm_training_data, **mmm_fit_kwargs)

In [ ]:
def add_period_average_lift(mmm, effects, baseline_panel, delivered_panel, window):
    if (
        len(window) == 0
        or not window.is_unique
        or not window.is_monotonic_increasing
        or not effects.window_start.eq(window[0]).fillna(False).all()
        or not effects.window_end.eq(window[-1]).fillna(False).all()
        or not effects.n_periods.eq(len(window)).fillna(False).all()
    ):
        raise ValueError("The spend window must match every lift row.")

    geo_positions = {g: i for i, g in enumerate(mmm.model.coords["geo"])}
    search_position = list(mmm.model.coords["channel"]).index("search")
    columns = [(g, "search") for g in effects.geo]
    before = baseline_panel.loc[window, columns].to_numpy().T / SCALE
    after = delivered_panel.loc[window, columns].to_numpy().T / SCALE
    indices = ptx.as_xtensor(
        [geo_positions[g] for g in effects.geo], dims=("lift_geo",)
    )
    with mmm.model:
        mmm.model.add_coord("lift_geo", effects.geo.to_list())
        mmm.model.add_coord("lift_week", window.to_list())
        lam = mmm.model["saturation_lam"].isel(geo=indices, channel=search_position)
        beta = mmm.model["saturation_beta"].isel(geo=indices, channel=search_position)
        before = ptx.as_xtensor(before, dims=("lift_geo", "lift_week"))
        after = ptx.as_xtensor(after, dims=("lift_geo", "lift_week"))
        weekly_lift = mmm.saturation.function(
            after, lam, beta
        ) - mmm.saturation.function(before, lam, beta)
        mean_lift = pmd.Deterministic(
            "multi_arm_geo_lift_model_estimated_lift", weekly_lift.mean(dim="lift_week")
        )
        pmd.Deterministic(
            "multi_arm_geo_lift_model_estimated_lift_original_scale", mean_lift * SCALE
        )
        pmd.Normal(
            "multi_arm_geo_lift",
            mu=mean_lift,
            sigma=ptx.as_xtensor(effects.sigma.to_numpy() / SCALE, dims=("lift_geo",)),
            observed=ptx.as_xtensor(
                effects.delta_y.to_numpy() / SCALE, dims=("lift_geo",)
            ),
        )


mmm_calibrated = make_mmm()
mmm_calibrated.build_model(mmm_training_data)
add_period_average_lift(
    mmm_calibrated, lift_rows, spend_baseline, spend_realized, window
)
np.testing.assert_equal(mmm_calibrated.model["multi_arm_geo_lift"].eval().size, 32)
idata_calibrated = mmm_calibrated.fit(mmm_training_data, **mmm_fit_kwargs)

Keep the lift rows and both spend paths and their date window to reconstruct the calibrated graph. Rebuild it from the canonical training data with `baseline_input`, then reattach the temporal likelihood. For stable spend pairs see the standard scalar API in {ref}`mmm_geolift_calibration`.

In [ ]:
def diagnostic_row(label, quantity, draws, sample_stats=None):
    return {
        "Fit": label,
        "Quantity": quantity,
        "Divergences": int(sample_stats.diverging.sum())
        if sample_stats is not None
        else np.nan,
        "Max R-hat": float(az.rhat(draws).to_array().max()),
        "Min bulk ESS": float(az.ess(draws).to_array().min()),
    }


fits = {"Uncalibrated": idata_uncalibrated, "Calibrated": idata_calibrated}
sc_idata = result.model.idata
diagnostics = [
    diagnostic_row(
        "CausalPy",
        "Model parameters",
        sc_idata.posterior.to_dataset(),
        sc_idata.sample_stats,
    ),
    diagnostic_row("CausalPy", "Mean effects", effect_draws.to_dataset(name="effect")),
]
for label, idata in fits.items():
    posterior = idata.posterior.to_dataset()
    model = mmm_uncalibrated.model if label == "Uncalibrated" else mmm_calibrated.model
    free_variables = [rv.name for rv in model.free_RVs]
    diagnostics.append(
        diagnostic_row(
            label, "All free parameters", posterior[free_variables], idata.sample_stats
        )
    )
    for channel in channels:
        for parameter, description in [("lam", "Curvature"), ("beta", "Amplitude")]:
            for suffix, level in [
                ("", "geo estimates"),
                ("_raw_mu", "log-population mean"),
                ("_raw_sigma", "log-population spread"),
            ]:
                variable = f"saturation_{parameter}{suffix}"
                diagnostics.append(
                    diagnostic_row(
                        label,
                        f"{channel.title()} {description.lower()} {level}",
                        posterior[[variable]].sel(channel=channel),
                    )
                )

display(
    pd.DataFrame(diagnostics)
    .set_index(["Fit", "Quantity"])
    .style.format(
        {"Divergences": "{:,.0f}", "Max R-hat": "{:.4f}", "Min bulk ESS": "{:,.0f}"},
        na_rep="—",
    )
)

Check zero divergences, R-hat near or below 1.01, and bulk ESS around 400 or more for quantities you intend to interpret. Each row reports the worst R-hat and ESS in its group. Slow mixing of a population spread limits claims about between-market variation even when a response curve mixes well.

### Recovery of channel contributions

How much revenue does search contribute across spend levels? Compare example curves selected by four baseline-spend ranks in each arm. The table covers all markets and assesses both accuracy and uncertainty.

In [ ]:
spend_grid = np.linspace(0, 80_000, 80)


def response_draws(idata):
    p = idata.posterior
    spend = xr.DataArray(spend_grid / SCALE, dims="spend", coords={"spend": spend_grid})
    return SCALE * p.saturation_beta * np.tanh(p.saturation_lam * spend / 2)


curve_draws = {
    "Uncalibrated": response_draws(idata_uncalibrated),
    "Calibrated": response_draws(idata_calibrated),
}


def plot_response_by_arm(channel, examples):
    example_geos = [g for _, g in examples]
    n_rows = len(examples) // 3
    posterior_upper = max(
        float(
            az.hdi(draws.sel(geo=example_geos, channel=channel), prob=0.94)
            .sel(ci_bound="upper")
            .max()
        )
        for draws in curve_draws.values()
    )
    c = channels.index(channel)
    truth_upper = SCALE * true_beta[[geos.index(g) for g in example_geos], c].max()
    curve_upper = 1.05 * max(posterior_upper, truth_upper)
    fig, axes = plt.subplots(
        n_rows,
        3,
        figsize=(FIG_WIDTH, FIG_HEIGHT * (1.1 * n_rows)),
        sharex=True,
        sharey=True,
    )
    for col, arm in enumerate(["up", "down", "control"]):
        members = [g for example_arm, g in examples if example_arm == arm]
        color = ARM_COLORS[arm] if channel == "search" else CHANNEL_COLORS[channel]
        for row, g in enumerate(members):
            ax = axes[row, col]
            training_values = X_train.loc[X_train.geo == g, channel]
            ax.axvspan(
                training_values.min(),
                training_values.max(),
                facecolor="none",
                edgecolor=COLOR_DONOR,
                hatch="///",
                linewidth=0,
                alpha=0.35,
                zorder=0,
            )
            for label, alpha, style in [
                ("Uncalibrated", 0.35, ":"),
                ("Calibrated", 1.0, "-"),
            ]:
                draws = curve_draws[label].sel(geo=g, channel=channel)
                hdi = az.hdi(draws, prob=0.94)
                ax.fill_between(
                    spend_grid,
                    hdi.sel(ci_bound="lower"),
                    hdi.sel(ci_bound="upper"),
                    color=color,
                    alpha=alpha * 0.2,
                )
                ax.plot(
                    spend_grid,
                    draws.mean(["chain", "draw"]),
                    color=color,
                    alpha=alpha,
                    linestyle=style,
                )
            if channel == "search" and arm in ["up", "down"]:
                before = spend_baseline.loc[window, (g, "search")]
                after = spend_realized.loc[window, (g, "search")]
                ax.axvspan(
                    min(before.min(), after.min()),
                    max(before.max(), after.max()),
                    color=COLOR_CONTROL,
                    alpha=0.15,
                    zorder=0,
                )
            else:
                ax.text(
                    0.04,
                    0.92,
                    "No direct lift test",
                    transform=ax.transAxes,
                    color=COLOR_CONTROL,
                    fontsize=10,
                    va="top",
                )
            i = geos.index(g)
            truth = (
                SCALE
                * true_beta[i, c]
                * np.tanh(true_lam[i, c] * spend_grid / SCALE / 2)
            )
            ax.plot(spend_grid, truth, color="black", linestyle="--")
            ax.set_title(f"{arm.title()} arm · {g}", fontsize=12, fontweight="bold")
            ax.xaxis.set_major_formatter(USD)
            ax.set_xticks([0, 40_000, 80_000])
            ax.yaxis.set_major_formatter(USD)
            ax.set(xlim=(0, 80_000), ylim=(0, curve_upper))
            if col == 0:
                ax.set_ylabel(f"{channel.title()} contribution\n(USD/week)")
            if row == len(members) - 1:
                ax.set_xlabel(f"{channel.title()} spend\n(USD/week)")
    legend_color = "#444444" if channel == "search" else CHANNEL_COLORS[channel]
    handles = [
        Line2D(
            [], [], color=legend_color, alpha=0.35, linestyle=":", label="Uncalibrated"
        ),
        Line2D([], [], color=legend_color, label="Calibrated"),
        Line2D([], [], color="black", linestyle="--", label="Truth"),
        Patch(
            facecolor="none",
            edgecolor=COLOR_DONOR,
            hatch="///",
            alpha=0.35,
            label="MMM training spend range",
        ),
    ]
    if channel == "search":
        handles.append(
            Patch(
                facecolor=COLOR_CONTROL,
                alpha=0.15,
                label="Weekly test-contrast envelope",
            )
        )
    fig.legend(handles=handles, loc="outside upper center", ncol=3, fontsize=10)
    plt.show()


plot_response_by_arm("search", curve_examples)

**Caption:** Search response curves for four baseline-spend-selected geos in each arm. Columns are up, down, and control, colored blue, orange, and gray respectively. Light/dotted curves are uncalibrated posterior means, dark/solid curves are calibrated means, bands are pointwise 94% HDIs, and black dashed curves are known truth. Hatched spans mark each geo's min–max training spend. Solid gray spans cover the envelope of all eight weekly baseline/delivered search-spend pairs.

Only the up and down columns have direct experimental search constraints. Control panels say "No direct lift test." Historical training ranges and experiment envelopes are separate references. The experiment constrains the average of eight weekly contrasts, not every point inside the shaded envelope.

The experiment constrains an **average of weekly response differences**, not absolute channel contributions. Curves can agree on that local difference while remaining vertically far apart. Control-market estimates can change indirectly through the shared model; compare their truth curves to assess recovery rather than treating an unchanged estimate as evidence of accuracy.

In [ ]:
metric_rows = []
for label, draws in curve_draws.items():
    hdi = az.hdi(draws, prob=0.94)
    for g in geos:
        i = geos.index(g)
        for channel in channels:
            c = channels.index(channel)
            selected = draws.sel(geo=g, channel=channel)
            true_curve = (
                SCALE
                * true_beta[i, c]
                * np.tanh(true_lam[i, c] * spend_grid / SCALE / 2)
            )
            bounds = hdi.sel(geo=g, channel=channel)
            metric_rows.append(
                {
                    "model": label,
                    "arm": geo_arm[g],
                    "channel": channel,
                    "MAE (USD/week)": float(
                        np.abs(selected.mean(["chain", "draw"]) - true_curve).mean()
                    ),
                    "HDI width (USD/week)": float(
                        (
                            bounds.sel(ci_bound="upper") - bounds.sel(ci_bound="lower")
                        ).mean()
                    ),
                    "Truth coverage": float(
                        (
                            (true_curve >= bounds.sel(ci_bound="lower"))
                            & (true_curve <= bounds.sel(ci_bound="upper"))
                        ).mean()
                    ),
                }
            )
curve_metrics = pd.DataFrame(metric_rows).groupby(["channel", "arm", "model"]).mean()
display(curve_metrics.xs("search", level="channel").round(2))

Grid summaries average over 0–80,000 USD/week, including extrapolation beyond historical spend. Coverage is the share of geo/grid truth values inside the pointwise 94% HDIs. The contribution-to-truth ratio below evaluates both curves at each market's operating spend.

In [ ]:
curve_diagnostics = []
for label, draws in curve_draws.items():
    central_curves = draws.sel(spend=spend_grid[[20, 40, 60]])
    curve_diagnostics.append(
        {
            "model": label,
            "curve max R-hat": float(az.rhat(central_curves).max()),
            "curve min ESS": float(az.ess(central_curves).min()),
        }
    )
display(pd.DataFrame(curve_diagnostics).round(4))

### Recovery of returns at operating spend

The experiment estimates average incremental revenue per dollar of delivered spend change. In a down geo, the ratio measures revenue lost per dollar cut. Marginal ROAS answers a different question: how much revenue does the next dollar generate at a particular spend level? For the logistic curve in original USD units,

$$
\operatorname{mROAS}_g(x)=f'_g(x)
=\frac{\beta_g\lambda_g}{2}\left[1-\tanh^2\!\left(\frac{\lambda_gx}{2S}\right)\right],
\qquad S=100{,}000.
$$

With a concave response, a down test measures a secant below that week's baseline spend, putting an upper bound on the derivative at baseline and higher spend. An up test measures above baseline, where a model can flatten while its response below the tested interval stays steep. This asymmetry is especially consequential if the history credits demand to search. Averaging eight secants across changing budgets further separates the experimental estimand from the derivative at mean baseline spend.

In [ ]:
def marginal_roas(lam, beta, spend):
    return beta * lam / 2 * (1 - np.tanh(lam * spend / (2 * SCALE)) ** 2)


lift_geo = xr.DataArray(
    lift_rows.geo.to_numpy(),
    dims="lift_geo",
    coords={"lift_geo": lift_rows.geo.to_numpy()},
)
weekly_before = xr.DataArray(
    search_baseline.to_numpy().T,
    dims=("lift_geo", "lift_week"),
    coords={"lift_geo": lift_rows.geo.to_numpy(), "lift_week": window},
)
weekly_after = xr.DataArray(
    search_delivered.to_numpy().T,
    dims=("lift_geo", "lift_week"),
    coords=weekly_before.coords,
)
spend_change = (weekly_after - weekly_before).mean("lift_week")
local_returns = {}
for label, idata in fits.items():
    p = idata.posterior.sel(geo=lift_geo, channel="search")
    before = (
        SCALE
        * p.saturation_beta
        * np.tanh(p.saturation_lam * weekly_before / (2 * SCALE))
    )
    after = (
        SCALE
        * p.saturation_beta
        * np.tanh(p.saturation_lam * weekly_after / (2 * SCALE))
    )
    local_returns[label] = (after - before).mean("lift_week") / spend_change

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(FIG_WIDTH, FIG_HEIGHT * 4.7), sharey=True)
local_upper = 0.0
for ax, (arm, g) in zip(axes.flat, return_examples, strict=True):
    dose = lift_rows.set_index("geo").loc[g]
    before = spend_baseline.loc[window, (g, "search")]
    after = spend_realized.loc[window, (g, "search")]
    lower, upper = min(before.min(), after.min()), max(before.max(), after.max())
    local_spend = np.linspace(max(0, lower - 10_000), upper + 10_000, 100)
    spend = xr.DataArray(local_spend, dims="spend")
    ax.axvspan(
        lower,
        upper,
        color=COLOR_CONTROL,
        alpha=0.15,
        label="Weekly test-contrast envelope",
    )
    for label, idata, alpha, style in [
        ("Uncalibrated", idata_uncalibrated, 0.35, ":"),
        ("Calibrated", idata_calibrated, 1.0, "-"),
    ]:
        p = idata.posterior.sel(geo=g, channel="search")
        draws = marginal_roas(p.saturation_lam, p.saturation_beta, spend)
        hdi = az.hdi(draws, prob=0.94)
        ax.fill_between(
            local_spend,
            hdi.sel(ci_bound="lower"),
            hdi.sel(ci_bound="upper"),
            color=ARM_COLORS[arm],
            alpha=alpha * 0.2,
        )
        ax.plot(
            local_spend,
            draws.mean(["chain", "draw"]),
            color=ARM_COLORS[arm],
            alpha=alpha,
            linestyle=style,
            label=label,
        )
        local_upper = max(local_upper, float(hdi.sel(ci_bound="upper").max()))
    i = geos.index(g)
    true_marginal = marginal_roas(true_lam[i, 0], true_beta[i, 0], local_spend)
    ax.plot(
        local_spend,
        true_marginal,
        color="black",
        linestyle="--",
        label="True marginal ROAS",
    )
    ax.hlines(
        true_lift[g] / dose.delta_x,
        lower,
        upper,
        color="black",
        linewidth=2,
        label="True experiment-average ROAS",
    )
    local_upper = max(local_upper, float(true_marginal.max()))
    ax.axvline(dose.x, color=ARM_COLORS[arm], linestyle=":", alpha=0.6)
    ax.set_title(f"{g} · {arm} · search")
    ax.set_xlabel("Search spend (USD/week)")
    ax.set_xlim(local_spend[0], local_spend[-1])
    ax.xaxis.set_major_formatter(USD)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:.2f}"))
for ax in axes[:, 0]:
    ax.set_ylabel("Marginal ROAS\n(USD revenue / USD spend)")
axes[0, 0].set_ylim(0, local_upper * 1.05)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncol=3, fontsize=10)
plt.show()

**Caption:** Local search marginal ROAS for four predetermined geos per active arm. Lines are posterior means with pointwise 94% HDIs; light dotted curves are uncalibrated and dark solid curves are calibrated. Black dashed curves are true derivatives. Gray spans enclose all eight weekly baseline/delivered spend pairs, and arm-colored vertical dotted lines mark mean baseline spend. Black horizontal segments show true experiment-average incremental ROAS; they are neither point derivatives nor a secant at mean spend.

The experiment constrains the average weekly response difference. Its derivative at mean spend is a separate recovery check. The tables below compare both against truth and retain the counterfactual and marginal-summary uncertainty limitations.

In [ ]:
def return_summary(draws, truth):
    error = draws.mean(sample_dims) - truth
    hdi = az.hdi(draws, prob=0.94)
    lower, upper = hdi.sel(ci_bound="lower"), hdi.sel(ci_bound="upper")
    return {
        "Signed error": float(error.mean()),
        "MAE": float(np.abs(error).mean()),
        "RMSE": float(np.sqrt((error**2).mean())),
        "Mean 94% HDI width": float((upper - lower).mean()),
        "94% HDI coverage": float(((lower <= truth) & (truth <= upper)).mean()),
    }


operating_spend = xr.DataArray(
    baseline[is_intervention].mean(axis=0),
    coords={"geo": geos, "channel": channels},
    dims=("geo", "channel"),
)
lam_truth = xr.DataArray(
    true_lam, coords=operating_spend.coords, dims=operating_spend.dims
)
beta_truth = xr.DataArray(
    true_beta, coords=operating_spend.coords, dims=operating_spend.dims
)
contribution_truth = (
    SCALE * beta_truth * np.tanh(lam_truth * operating_spend / (2 * SCALE))
)
marginal_truth = marginal_roas(lam_truth, beta_truth, operating_spend)
baseline_rows = []
for label, idata in fits.items():
    p = idata.posterior
    contributions = (
        SCALE
        * p.saturation_beta
        * np.tanh(p.saturation_lam * operating_spend / (2 * SCALE))
    )
    returns = marginal_roas(p.saturation_lam, p.saturation_beta, operating_spend)
    for arm, members in arms.items():
        for channel in channels:
            key = {"geo": members, "channel": channel}
            baseline_rows.append(
                {"Fit": label, "Arm": arm, "Channel": channel}
                | return_summary(returns.sel(key), marginal_truth.sel(key))
                | {
                    "Mean contribution / truth": float(
                        (
                            contributions.sel(key).mean(sample_dims)
                            / contribution_truth.sel(key)
                        ).mean()
                    )
                }
            )
baseline_metrics = pd.DataFrame(baseline_rows).set_index(["Channel", "Arm", "Fit"])
display(baseline_metrics.xs("search", level="Channel").round(3))

In [ ]:
# Check the experimental contrast against the model's registered deterministic.
registered_lift = idata_calibrated.posterior[
    "multi_arm_geo_lift_model_estimated_lift_original_scale"
]
np.testing.assert_allclose(
    (local_returns["Calibrated"] * spend_change)
    .transpose("chain", "draw", "lift_geo")
    .values,
    registered_lift.transpose("chain", "draw", ...).values,
)


true_interval_roas = (
    xr.DataArray(
        true_lift.loc[lift_rows.geo].to_numpy(), dims="lift_geo", coords=lift_geo.coords
    )
    / spend_change
)
interval_rows = []
for label, draws in local_returns.items():
    for arm in ("up", "down"):
        members = arms[arm]
        interval_rows.append(
            {"Fit": label, "Arm": arm}
            | return_summary(
                draws.sel(lift_geo=members),
                true_interval_roas.sel(lift_geo=members),
            )
        )
display(pd.DataFrame(interval_rows).set_index(["Arm", "Fit"]).round(3))

Both tables report returns in USD of revenue per USD of spend. Signed error shows over- or underestimation. Coverage counts how often the 94% HDI contains a geo's true return. The contribution ratio averages the estimated-to-true search contribution ratios at each market’s operating spend.

### Recovery of the untested display channel

Display has no lift observations, but estimates can shift as the MMM reallocates revenue between media and the weekly baseline. Both display plots use the same nine example markets chosen by baseline display spend. The tables cover all 48 markets.

In [ ]:
plot_response_by_arm("display", display_examples)

**Caption:** Display response curves for three baseline-display-spend-selected geos per arm, with up, down, and control columns. Green identifies display; light/dotted and dark/solid posterior means denote uncalibrated and calibrated fits, with pointwise 94% HDI bands. Black dashed curves are known truth. Hatched spans mark each geo's min–max display training spend. Every panel has no direct lift test.

The calibrated display estimates can move even though only search is experimentally constrained. This does not establish improved display attribution: the table below checks curve error and uncertainty against truth across all geos.

These display summaries use the same grid, error, interval-width, and pointwise-coverage definitions as the search table. Changes in an untested channel remain dependent on the joint model and imperfect separation of the shared baseline from media response; narrower intervals alone do not establish better recovery.

The display changes are small and mixed: the control-arm grid error decreases slightly, while the active-arm errors increase slightly. These movements are small compared with the posterior bands. They do not establish consistent recovery gains for the untested channel.

In [ ]:
display(curve_metrics.xs("display", level="channel").round(2))

#### Display marginal ROAS at operating spend

Use the same derivative for display. Each vertical reference marks that market's mean business-as-usual display spend during the experiment.

In [ ]:
fig, axes = plt.subplots(
    3, 3, figsize=(FIG_WIDTH, FIG_HEIGHT * 3.3), sharex=True, sharey=True
)
display_return_upper = 0.0
color = CHANNEL_COLORS["display"]
for col, arm in enumerate(["up", "down", "control"]):
    members = [g for a, g in display_examples if a == arm]
    for row, g in enumerate(members):
        ax = axes[row, col]
        training_values = X_train.loc[X_train.geo == g, "display"]
        ax.axvspan(
            training_values.min(),
            training_values.max(),
            facecolor="none",
            edgecolor=COLOR_DONOR,
            hatch="///",
            linewidth=0,
            alpha=0.35,
            zorder=0,
        )
        spend = xr.DataArray(spend_grid, dims="spend", coords={"spend": spend_grid})
        for label, idata, alpha, style in [
            ("Uncalibrated", idata_uncalibrated, 0.35, ":"),
            ("Calibrated", idata_calibrated, 1.0, "-"),
        ]:
            params = idata.posterior.sel(geo=g, channel="display")
            draws = marginal_roas(params.saturation_lam, params.saturation_beta, spend)
            hdi = az.hdi(draws, prob=0.94)
            ax.fill_between(
                spend_grid,
                hdi.sel(ci_bound="lower"),
                hdi.sel(ci_bound="upper"),
                color=color,
                alpha=alpha * 0.2,
            )
            ax.plot(
                spend_grid,
                draws.mean(["chain", "draw"]),
                color=color,
                alpha=alpha,
                linestyle=style,
                label=label,
            )
            display_return_upper = max(
                display_return_upper, float(hdi.sel(ci_bound="upper").max())
            )
        i = geos.index(g)
        true_marginal = marginal_roas(true_lam[i, 1], true_beta[i, 1], spend_grid)
        ax.plot(spend_grid, true_marginal, color="black", linestyle="--")
        display_return_upper = max(display_return_upper, float(true_marginal.max()))
        ax.axvline(
            spend_baseline.loc[window, (g, "display")].mean(),
            color=color,
            linestyle=":",
            alpha=0.6,
        )
        ax.text(
            0.04,
            0.92,
            "No direct lift test",
            transform=ax.transAxes,
            color=COLOR_CONTROL,
            fontsize=10,
            va="top",
        )
        ax.set_title(f"{arm.title()} arm · {g}", fontsize=12, fontweight="bold")
        ax.set_xlim(0, 80_000)
        ax.set_xticks([0, 40_000, 80_000])
        ax.xaxis.set_major_formatter(USD)
        ax.yaxis.set_major_formatter(StrMethodFormatter("{x:.2f}"))
        if col == 0:
            ax.set_ylabel("Display marginal ROAS\n(USD revenue / USD spend)")
        if row == 2:
            ax.set_xlabel("Display spend\n(USD/week)")
axes[0, 0].set_ylim(0, display_return_upper * 1.05)
fig.legend(
    handles=[
        Line2D([], [], color=color, alpha=0.35, linestyle=":", label="Uncalibrated"),
        Line2D([], [], color=color, label="Calibrated"),
        Line2D([], [], color="black", linestyle="--", label="Truth"),
        Patch(
            facecolor="none",
            edgecolor=COLOR_DONOR,
            hatch="///",
            alpha=0.35,
            label="MMM training spend range",
        ),
        Line2D([], [], color=color, linestyle=":", alpha=0.6, label="Baseline spend"),
    ],
    loc="outside upper center",
    ncol=3,
    fontsize=10,
)
plt.show()

**Caption:** Display marginal-ROAS curves for the same nine geos as the display saturation figure, arranged in up, down, and control columns. Lines are posterior means with pointwise 94% HDI bands; light/dotted is uncalibrated, dark/solid is calibrated, and black dashed is truth. Vertical green dotted lines mark mean business-as-usual display spend during the experiment; hatched spans show min–max training spend. There is no direct display lift test in any panel.

In [ ]:
display(baseline_metrics.xs("display", level="Channel").round(3))

### How did the response parameters move?

Compare curvature by arm and each channel’s log-population spreads across the prior, both posteriors, and the simulated values. The spread is fit across all geos within a channel.

In [ ]:
prior_variables = [
    "saturation_lam",
    "saturation_lam_raw_sigma",
    "saturation_beta_raw_sigma",
]
with mmm_uncalibrated.model:
    response_prior = pm.sample_prior_predictive(
        draws=1_000, var_names=prior_variables, random_seed=SEED
    ).prior

prior_rows = []

for label, draws in {
    "Prior": response_prior,
    **{name: fit.posterior for name, fit in fits.items()},
}.items():
    for channel in channels:
        for arm, members in arms.items():
            curvature = draws.saturation_lam.sel(geo=members, channel=channel)
            prior_rows.append(
                {
                    "Channel": channel,
                    "Arm": arm,
                    "Fit": label,
                    "Mean curvature": float(curvature.mean()),
                    "Largest geo mean": float(curvature.mean(sample_dims).max()),
                    "True mean curvature": float(
                        lam_truth.sel(geo=members, channel=channel).mean()
                    ),
                }
            )
display(pd.DataFrame(prior_rows).set_index(["Channel", "Arm", "Fit"]).round(2))

spread_rows = []
for variable, truth_values, description in [
    ("saturation_lam_raw_sigma", lam_truth, "Log-curvature"),
    ("saturation_beta_raw_sigma", beta_truth, "Log-amplitude"),
]:
    for channel in channels:
        for label, draws in {
            "Prior": response_prior,
            **{name: fit.posterior for name, fit in fits.items()},
        }.items():
            spread = draws[variable].sel(channel=channel)
            spread_rows.append(
                {
                    "Channel": channel,
                    "Spread": description,
                    "Fit": label,
                    "Median": float(spread.median()),
                    "3%": float(spread.quantile(0.03)),
                    "97%": float(spread.quantile(0.97)),
                    "Realized log-SD": float(
                        np.log(truth_values.sel(channel=channel)).std("geo")
                    ),
                }
            )
display(pd.DataFrame(spread_rows).set_index(["Channel", "Spread", "Fit"]).round(2))

Where contributions are far from truth or interval coverage is poor, the downstream response remains unreliable even if experiment-average return improves. A posterior curvature or amplitude spread well away from its prior and known simulation value can flag tension between historical attribution, the experimental constraints, and the assumed response shape. Investigate that tension before using the curve for a budget decision.

## Summary and limitations

Use a shared time effect to adjust for common additive shocks when the panel and causal structure justify it. Then use the signed experiment to learn which parts of the response curve the history supports. Compare experiment-average returns, point derivatives, and contributions separately by arm. The experiment constrains the tested ranges. Control-search and display changes come through the joint attribution model.

The donor estimate depends on credible post-period transfer, no spillovers, and no concurrent shocks aligned with assignment. The fitted residual scale assumes independent weekly errors. The Normal handoff uses one mean and SD per geo and leaves out cross-market covariance and dependence from reusing the historical outcomes. This simulation uses an identity link and no carryover. With carryover, the experimental contrast must account for the full adstock history.

:::{seealso}

- {ref}`mmm_geolift_calibration` introduces the fixed-spend calibration API.
- {ref}`mmm_multidimensional_example` explains geographic dimensions and hierarchical priors.
- [CausalPy's up/down/control example](https://causalpy.readthedocs.io/en/latest/notebooks/up-down-control-geolift-pymc.html) develops experimental effect estimation.
:::

In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w -p pymc_marketing,causalpy,pytensor